# 01 langchain vs langgraph: practice

Messages → A tool schema describes; a dispatcher executes → Build a real local LangGraph without a model → Conditional routing makes the next step explicit

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [4_langgraph/community_contributions/1_basics/01_langchain_vs_langgraph.ipynb](../../../../../../../4_langgraph/community_contributions/1_basics/01_langchain_vs_langgraph.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Messages: instructions, earlier turns, and the new question

The model request is a list of messages. A role says who is speaking; content holds the text. The system instruction and prior conversation serve different purposes.

**Predict:** How many messages are sent on the follow-up turn?

In [ ]:
instructions = "Explain Python with small examples."
history = [
    {"role": "user", "content": "What is a list?"},
    {"role": "assistant", "content": "An ordered collection of values."},
]
# List addition preserves the order: instruction, old turns, new question.
messages = [{"role": "system", "content": instructions}] + history + [
    {"role": "user", "content": "How is a dictionary different?"}
]
print([item["role"] for item in messages])
assert len(messages) == 4

**Check:** Four: system, user, assistant, user. With empty history, only two remain.

**Change one thing:** Set history to an empty list. Explain what is lost while the system instruction remains.

## 2. A tool schema describes; a dispatcher executes

The model can request a function call. Your application decodes arguments, recognizes the name, executes the function, and returns a matching tool result. No model is needed to practice this boundary.

**Predict:** Does defining the schema call the function?

In [ ]:
import json
records = []
def record_interest(email):
    records.append(email)  # In memory only; does not send email or notifications.
    return {"recorded": True}
schema = {
    "type": "function", "function": {
        "name": "record_interest", "description": "Record a supplied address",
        "parameters": {"type": "object", "properties": {"email": {"type": "string"}},
                       "required": ["email"], "additionalProperties": False},
    },
}
assert records == []
requested = {"id": "call_1", "name": "record_interest", "arguments": '{"email":"learner@example.com"}'}
allowed = {"record_interest": record_interest}  # An explicit registry of permitted functions.
if requested["name"] not in allowed:
    raise ValueError("Unknown tool")
result = allowed[requested["name"]](**json.loads(requested["arguments"]))
tool_message = {"role": "tool", "tool_call_id": requested["id"], "content": json.dumps(result)}
print(tool_message)
assert records == ["learner@example.com"]

**Check:** No. Only the explicit dispatcher call changes records. Keep the assistant tool-call message before the matching tool result in a full conversation.

**Change one thing:** Change the requested name to an unknown function and observe the error. Then add a harmless second tool to the registry.

## 3. Build a real local LangGraph without a model

A graph is state plus nodes plus edges. Nodes return state updates. Compilation builds the executable workflow; invoke runs it. This exercises the installed LangGraph package locally.

**Predict:** Which value will the uppercase node return?

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
class State(TypedDict):
    text: str
def uppercase(state):
    return {"text": state["text"].upper()}
builder = StateGraph(State)
builder.add_node("uppercase", uppercase)
builder.add_edge(START, "uppercase")
builder.add_edge("uppercase", END)
graph = builder.compile()
result = graph.invoke({"text": "hello"})
print(result)
assert result["text"] == "HELLO"

**Check:** HELLO. Graphs can orchestrate ordinary Python functions; a language model is optional.

**Change one thing:** Add a second node that appends an exclamation mark. Wire it between uppercase and END.

## 4. Conditional routing makes the next step explicit

A route is a function of state. A worker may need tools, evaluation, another attempt, or user clarification. This exercise checks all branch labels without calling any model.

**Predict:** Which route wins when a tool call is present?

In [ ]:
def worker_route(state):
    return "tools" if state["tool_calls"] else "evaluator"
def evaluation_route(state):
    if state["success"] or state["needs_user"]:
        return "END"
    return "worker"
assert worker_route({"tool_calls": ["lookup"]}) == "tools"
assert worker_route({"tool_calls": []}) == "evaluator"
for state in [{"success": True, "needs_user": False},
              {"success": False, "needs_user": True},
              {"success": False, "needs_user": False}]:
    print(evaluation_route(state))

**Check:** Tools first. Evaluation can stop for success or required user input, and otherwise return to the worker.

**Change one thing:** Add an attempts field and route to END after a fixed limit. Explain why asking for clarification is a valid stop condition.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.